## **Laboratorio: RAG Avanzado — Recuperación, Re-ranking y Evaluación**

El RAG básico (embeddings + prompt) resuelve el 20% del problema. Los sistemas de recuperación en producción requieren siete técnicas adicionales:

1. **Chunking consciente de tokens**: dividir documentos respetando límites semánticos, con solapamiento controlado.
2. **Retrieval híbrido**: combinar búsqueda semántica (densa) con búsqueda léxica (BM25 dispersa) para capturar coincidencias exactas *y* semánticas.
3. **Fusión de rankings (RRF)**: fusionar listas de resultados heterogéneos mediante Reciprocal Rank Fusion.
4. **Re-ranking con Cross-Encoder**: un modelo más potente pero lento re-puntúa los candidatos finalistas.
5. **Generación con grounding y citas**: forzar al LLM a fundamentar cada afirmación en fragmentos recuperados.
6. **Evaluación multidimensional**: relevancia de recuperación, fidelidad de la respuesta, latencia y coste.
7. **Filtrado por metadatos**: restringir la búsqueda a versiones, fechas o temas concretos.

---

Este laboratorio implementa un pipeline completo end-to-end sobre un corpus de 15 documentos técnicos en español sobre Inteligencia Artificial y Machine Learning.

### ¿Qué practicarás?

En este laboratorio, conectando con los contenidos del máster, implementarás desde cero:

- **Limpieza y chunking con tiktoken**: normalización de texto y división en fragmentos de tamaño controlado con solapamiento.
- **Índice vectorial con ChromaDB**: almacenamiento y recuperación eficiente de embeddings con filtros de metadatos.
- **Búsqueda semántica y BM25**: dos paradigmas de recuperación complementarios.
- **Reciprocal Rank Fusion**: algoritmo matemático para combinar rankings heterogéneos.
- **Cross-Encoder para re-ranking**: modelo sequence-pair que evalúa relevancia pregunta-fragmento.
- **Generación RAG con citas numeradas**: respuestas fundamentadas con referencias explícitas a fuentes.
- **Evaluación automática**: cosine similarity, LLM-as-judge para fidelidad, y métricas de coste.

### Objetivos

Al finalizar este laboratorio, serás capaz de:

- Construir un pipeline RAG completo con más de cinco etapas encadenadas.
- Explicar las diferencias entre retrieval denso (semántico) y disperso (BM25), y cuándo usar cada uno.
- Implementar Reciprocal Rank Fusion y justificar matemáticamente su ventaja sobre la simple unión de listas.
- Aplicar un Cross-Encoder para re-ranking y medir cuánto mejora la calidad de recuperación.
- Diseñar prompts de sistema que fuercen grounding y citas, reduciendo el riesgo de alucinación.
- Medir relevancia de recuperación con similitud coseno, detectar alucinaciones con LLM-as-judge, y calcular el coste real por consulta.
- Identificar y demostrar los cuatro errores más comunes en sistemas RAG en producción.

## Sección 1: Configuración del Entorno

**Celda 1: Instalación de dependencias**

In [ ]:
!pip install openai chromadb rank_bm25 sentence-transformers tiktoken

Instalamos las cinco librerías del stack:
- **openai**: cliente oficial para la Responses API y los embeddings de OpenAI.
- **chromadb**: base de datos vectorial local con soporte para metadatos y filtros.
- **rank_bm25**: implementación de BM25Okapi para búsqueda léxica clásica.
- **sentence-transformers**: modelos de re-ranking basados en transformers (Cross-Encoder).
- **tiktoken**: tokenizador de OpenAI, necesario para chunking consciente de tokens.

**Celda 2: Configuración de la clave de API**

In [ ]:
OPENAI_API_KEY = "sk-..."  # ← Reemplaza con tu clave real

La clave de API autentica tus peticiones contra los servidores de OpenAI. **Nunca la subas a un repositorio público.** En producción, usa variables de entorno (`os.environ["OPENAI_API_KEY"]`) o gestores de secretos como AWS Secrets Manager o HashiCorp Vault.

**Celda 3: Importaciones y cliente OpenAI**

In [ ]:
import openai
import chromadb
import tiktoken
import time
import json
import re
import math
import numpy as np
import pandas as pd

from openai import OpenAI
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

# Cliente OpenAI
client = OpenAI(api_key=OPENAI_API_KEY)

print("✓ Todas las librerías importadas correctamente.")
print(f"  OpenAI SDK versión : {openai.__version__}")
print(f"  ChromaDB versión   : {chromadb.__version__}")
print(f"  NumPy versión      : {np.__version__}")
print(f"  Pandas versión     : {pd.__version__}")

Importamos todo en una sola celda para tener un único punto de control de dependencias. Crear el cliente `OpenAI` aquí garantiza que cualquier función posterior pueda reutilizarlo sin reinicializarlo.

## Sección 2: Corpus de Documentos

**Celda 4: Definición del corpus de 15 documentos**

In [ ]:
documentos = [
    {
        "id": "doc_01",
        "titulo": "Fundamentos del Aprendizaje Automático",
        "contenido": (
            "El aprendizaje automático (machine learning) es una subdisciplina de la inteligencia artificial "
            "que permite a los sistemas aprender y mejorar a partir de la experiencia sin ser programados "
            "explícitamente. En lugar de seguir instrucciones rígidas, los algoritmos de ML identifican "
            "patrones en los datos y construyen modelos matemáticos para hacer predicciones o tomar decisiones.\n\n"
            "Existen tres paradigmas principales. El aprendizaje supervisado utiliza datos etiquetados: "
            "el algoritmo aprende la correspondencia entre entradas (features) y salidas (etiquetas) "
            "mediante un proceso de optimización que minimiza una función de pérdida. Ejemplos típicos "
            "son la regresión lineal, los árboles de decisión, las máquinas de vectores de soporte (SVM) "
            "y los bosques aleatorios (Random Forest).\n\n"
            "El aprendizaje no supervisado trabaja con datos sin etiquetar y busca estructura latente: "
            "clustering con K-Means o DBSCAN, reducción de dimensionalidad con PCA o t-SNE, y modelos "
            "generativos. El aprendizaje por refuerzo, por su parte, optimiza una política de acción "
            "mediante recompensas y penalizaciones, inspirándose en la psicología conductista.\n\n"
            "El pipeline estándar de ML comprende: recolección de datos, exploración (EDA), preprocesado "
            "(normalización, codificación, imputación), selección de características, entrenamiento, "
            "validación cruzada, ajuste de hiperparámetros y evaluación final en el conjunto de prueba. "
            "La separación estricta entre train, validation y test es fundamental para evitar el sobreajuste "
            "y garantizar que las métricas reportadas sean representativas del rendimiento real en producción."
        ),
        "fuente": "Manual de ML v1.0",
        "fecha": "2023-01-15",
        "version": "1.0",
        "tema": "machine_learning"
    },
    {
        "id": "doc_02",
        "titulo": "Redes Neuronales Artificiales: Arquitectura y Entrenamiento",
        "contenido": (
            "Las redes neuronales artificiales (ANN) son modelos computacionales inspirados en la estructura "
            "del cerebro biológico. Están compuestas por nodos (neuronas artificiales) organizados en capas "
            "interconectadas: una capa de entrada, una o varias capas ocultas y una capa de salida. Cada "
            "conexión tiene un peso ajustable que determina la influencia de una neurona sobre otra.\n\n"
            "La unidad básica aplica una transformación afín seguida de una función de activación no lineal. "
            "Las funciones de activación más utilizadas son ReLU (Rectified Linear Unit), que resuelve el "
            "problema del gradiente evanescente en capas profundas; Sigmoid, usada en salidas binarias; "
            "Softmax, para clasificación multiclase; y GELU, adoptada en transformers modernos.\n\n"
            "El entrenamiento se realiza mediante el algoritmo de retropropagación (backpropagation): se "
            "calcula el gradiente de la función de pérdida respecto a cada peso usando la regla de la cadena, "
            "y se actualizan los pesos con descenso de gradiente estocástico (SGD) o variantes adaptativas "
            "como Adam, AdaGrad o RMSProp. El hiperparámetro tasa de aprendizaje (learning rate) es crítico: "
            "demasiado alto causa divergencia; demasiado bajo, convergencia lenta.\n\n"
            "Técnicas de regularización como Dropout (desactivar neuronas aleatoriamente durante el "
            "entrenamiento), Batch Normalization (normalizar activaciones por mini-batch) y Weight Decay "
            "(penalización L2) son esenciales para evitar el sobreajuste en redes profundas. La arquitectura "
            "óptima depende de la tarea: capas densas para datos tabulares, convolucionales para imágenes "
            "y recurrentes o de atención para secuencias."
        ),
        "fuente": "Fundamentos de Deep Learning",
        "fecha": "2023-02-20",
        "version": "1.0",
        "tema": "redes_neuronales"
    },
    {
        "id": "doc_03",
        "titulo": "Procesamiento del Lenguaje Natural: De TF-IDF a BERT",
        "contenido": (
            "El procesamiento del lenguaje natural (NLP) abarca las técnicas que permiten a las máquinas "
            "comprender, generar y manipular texto humano. La evolución histórica del campo refleja el "
            "progreso general del machine learning: de métodos estadísticos clásicos a redes profundas.\n\n"
            "Los enfoques clásicos representaban el texto como vectores de frecuencia de términos. TF-IDF "
            "(Term Frequency–Inverse Document Frequency) pondera cada término por su frecuencia en el "
            "documento y su rareza en el corpus, produciendo representaciones dispersas de alta dimensión. "
            "Los modelos de n-gramas capturan dependencias locales, pero son incapaces de generalizar a "
            "combinaciones no vistas.\n\n"
            "La revolución de los embeddings comenzó con Word2Vec (2013): una red neuronal superficial que "
            "aprende representaciones densas de palabras a partir del contexto local (ventana deslizante). "
            "Palabras semánticamente similares quedan próximas en el espacio vectorial. GloVe y FastText "
            "ampliaron este paradigma. Sin embargo, estos embeddings son estáticos: la palabra 'banco' "
            "tiene el mismo vector independientemente de si habla de dinero o de madera.\n\n"
            "ELMo introdujo embeddings contextuales mediante LSTMs bidireccionales. La verdadera ruptura "
            "llegó con BERT (2018): un transformer pre-entrenado con enmascaramiento de tokens (MLM) y "
            "predicción de siguiente oración (NSP). BERT produce embeddings contextuales de 768 dimensiones "
            "que capturan matices semánticos dependientes del contexto. El fine-tuning de BERT sobre tareas "
            "específicas (clasificación, NER, QA) batió los benchmarks de la época con mejoras de más del 10%."
        ),
        "fuente": "NLP Avanzado - Módulo 3",
        "fecha": "2023-03-10",
        "version": "1.0",
        "tema": "nlp"
    },
    {
        "id": "doc_04",
        "titulo": "Visión por Computador y Redes Convolucionales",
        "contenido": (
            "La visión por computador (Computer Vision) permite a las máquinas interpretar información visual: "
            "imágenes y vídeo. Las redes neuronales convolucionales (CNN) son la arquitectura dominante para "
            "tareas visuales gracias a su capacidad para aprender jerarquías de características espaciales.\n\n"
            "Una CNN aplica filtros (kernels) convolucionales que detectan patrones locales: bordes y texturas "
            "en capas tempranas, partes de objetos en capas intermedias y objetos completos en capas profundas. "
            "El operador de convolución discreta calcula el producto punto entre el filtro y cada región local "
            "de la imagen, produciendo un mapa de características (feature map). El stride controla el "
            "desplazamiento del filtro y el padding preserva las dimensiones espaciales.\n\n"
            "Las capas de pooling (MaxPool, AvgPool) reducen la resolución espacial, aportando invarianza "
            "traslacional y reduciendo la carga computacional. La arquitectura clásica AlexNet (2012) "
            "demostró que las CNNs profundas superaban los métodos manuales de extracción de características. "
            "Arquitecturas posteriores como VGG, ResNet (conexiones residuales para gradientes más profundos), "
            "Inception y EfficientNet mejoraron sistemáticamente la eficiencia y precisión.\n\n"
            "Las aplicaciones incluyen clasificación de imágenes, detección de objetos (YOLO, Faster R-CNN), "
            "segmentación semántica (U-Net), estimación de pose y reconocimiento facial. La transferencia de "
            "aprendizaje (transfer learning) permite adaptar CNNs pre-entrenadas en ImageNet a nuevos dominios "
            "con pocos datos adicionales, reduciendo drásticamente el tiempo y coste de entrenamiento."
        ),
        "fuente": "Computer Vision - Capítulo 5",
        "fecha": "2023-04-05",
        "version": "1.0",
        "tema": "vision_computador"
    },
    {
        "id": "doc_05",
        "titulo": "Aprendizaje por Refuerzo: De Q-Learning a PPO",
        "contenido": (
            "El aprendizaje por refuerzo (RL) es el paradigma donde un agente aprende a tomar decisiones "
            "en un entorno para maximizar una señal de recompensa acumulada. A diferencia del aprendizaje "
            "supervisado, no hay etiquetas correctas: el agente descubre qué acciones son buenas mediante "
            "exploración y explotación.\n\n"
            "El marco formal es el Proceso de Decisión de Markov (MDP): un agente en el estado s toma "
            "la acción a, recibe recompensa r y transiciona al estado s'. El objetivo es aprender una "
            "política π(a|s) que maximice el retorno esperado (suma descontada de recompensas futuras). "
            "El factor de descuento γ pondera las recompensas inmediatas frente a las futuras.\n\n"
            "Q-Learning aprende la función de valor de acción Q(s,a) mediante la ecuación de Bellman: "
            "Q(s,a) ← Q(s,a) + α[r + γ·max_a' Q(s',a') - Q(s,a)]. Deep Q-Network (DQN, DeepMind 2015) "
            "usa una red neuronal para aproximar Q, con replay buffer y target network para estabilizar "
            "el entrenamiento. Batió a expertos humanos en 49 juegos de Atari.\n\n"
            "Los métodos de gradiente de política (Policy Gradient) optimizan directamente π. REINFORCE "
            "calcula el gradiente exacto pero tiene alta varianza. Actor-Critic combina un actor (política) "
            "con un crítico (función de valor) para reducir la varianza. PPO (Proximal Policy Optimization) "
            "añade una restricción que impide actualizaciones demasiado grandes, logrando estabilidad y "
            "eficiencia. PPO es el algoritmo detrás del RLHF que alinea los LLMs modernos con preferencias humanas."
        ),
        "fuente": "RL Moderno - Edición 2023",
        "fecha": "2023-05-12",
        "version": "1.0",
        "tema": "reinforcement_learning"
    },
    {
        "id": "doc_06",
        "titulo": "Transformers: Arquitectura de Atención Multi-Cabeza",
        "contenido": (
            "La arquitectura Transformer, propuesta en 'Attention is All You Need' (Vaswani et al., 2017), "
            "ha revolucionado el procesamiento de secuencias al eliminar la recurrencia y basar todo el "
            "procesamiento en mecanismos de atención diferenciable.\n\n"
            "El mecanismo de atención escalada calcula la relevancia entre cada par de posiciones en una "
            "secuencia. Dado un conjunto de consultas Q, claves K y valores V (matrices obtenidas por "
            "proyecciones lineales aprendidas del embedding de entrada), la atención se calcula como: "
            "Attention(Q,K,V) = softmax(QK^T / √d_k) · V. El factor √d_k estabiliza los gradientes "
            "cuando la dimensión d_k es grande.\n\n"
            "La atención multi-cabeza (Multi-Head Attention) ejecuta h instancias paralelas de atención, "
            "cada una en un subespacio de menor dimensión: MultiHead(Q,K,V) = Concat(head_1,...,head_h)·W^O. "
            "Esto permite que el modelo atienda simultáneamente a información de diferentes posiciones "
            "y diferentes subespacios de representación.\n\n"
            "La arquitectura completa del encoder apila N bloques idénticos, cada uno con: (1) atención "
            "multi-cabeza con conexión residual y normalización por capas, (2) red feed-forward posición a "
            "posición con conexión residual y normalización. El positional encoding añade información sobre "
            "la posición de cada token mediante funciones sinusoidales o embeddings aprendidos.\n\n"
            "El decoder incluye adicionalmente atención cruzada (cross-attention) sobre la salida del "
            "encoder, y la atención propia está enmascarada causalmente para impedir ver tokens futuros. "
            "Esta arquitectura es la base de GPT, BERT, T5 y todos los LLMs modernos."
        ),
        "fuente": "Transformers en Profundidad",
        "fecha": "2023-06-01",
        "version": "1.0",
        "tema": "transformers"
    },
    {
        "id": "doc_07",
        "titulo": "Fine-Tuning de Modelos de Lenguaje",
        "contenido": (
            "El fine-tuning consiste en continuar el entrenamiento de un modelo pre-entrenado sobre un "
            "conjunto de datos específico del dominio o tarea objetivo. Aprovecha el conocimiento general "
            "adquirido durante el pre-entrenamiento (semántica, gramática, razonamiento básico) y lo "
            "especializa para casos de uso concretos con mucha menos computación.\n\n"
            "El fine-tuning completo (full fine-tuning) actualiza todos los parámetros del modelo. Esto "
            "maximiza la adaptación pero requiere la misma memoria que el entrenamiento original y puede "
            "causar olvido catastrófico (catastrophic forgetting) del conocimiento general. Para LLMs con "
            "miles de millones de parámetros, es computacionalmente prohibitivo sin hardware de gama alta.\n\n"
            "Los métodos de fine-tuning eficiente en parámetros (PEFT) resuelven estas limitaciones. "
            "LoRA (Low-Rank Adaptation) descompone las actualizaciones de peso en matrices de rango bajo: "
            "ΔW = BA, donde B ∈ R^{d×r} y A ∈ R^{r×k} con r << min(d,k). Solo se entrenan A y B, "
            "reduciendo los parámetros entrenables en un 99% manteniendo un rendimiento similar al "
            "fine-tuning completo. QLoRA combina LoRA con cuantización de 4 bits para ejecutar en "
            "GPUs de consumo.\n\n"
            "El instruction tuning entrena el modelo en pares (instrucción, respuesta deseable), mejorando "
            "la capacidad de seguir instrucciones. RLHF (RL from Human Feedback) añade una fase de "
            "alineación con preferencias humanas: primero se entrena un modelo de recompensa sobre "
            "comparaciones humanas, luego se optimiza la política con PPO para maximizar esa recompensa, "
            "produciendo modelos más útiles, inofensivos y honestos como GPT-4 o Claude."
        ),
        "fuente": "Fine-Tuning de LLMs - Guía Práctica",
        "fecha": "2023-07-18",
        "version": "1.0",
        "tema": "fine_tuning"
    },
    {
        "id": "doc_08",
        "titulo": "RAG: Retrieval-Augmented Generation",
        "contenido": (
            "RAG (Retrieval-Augmented Generation) es una arquitectura que combina la capacidad generativa "
            "de los LLMs con la recuperación de información de fuentes externas, reduciendo alucinaciones "
            "y permitiendo actualizar el conocimiento sin reentrenamiento.\n\n"
            "El pipeline RAG estándar tiene tres fases. En la fase de indexación, los documentos se "
            "dividen en fragmentos (chunks), se generan embeddings para cada fragmento y se almacenan en "
            "una base de datos vectorial con sus metadatos. En la fase de recuperación, la consulta del "
            "usuario se embebe con el mismo modelo, se buscan los fragmentos más similares por distancia "
            "coseno o producto escalar, y se recuperan los top-k candidatos. En la fase de generación, "
            "los fragmentos recuperados se incluyen en el contexto del LLM junto con la pregunta, y el "
            "modelo genera una respuesta fundamentada en esa evidencia.\n\n"
            "Las limitaciones del RAG básico han impulsado técnicas avanzadas: retrieval híbrido (semántico "
            "+ BM25), re-ranking con cross-encoders, RAG iterativo (el modelo genera consultas de seguimiento), "
            "HyDE (Hypothetical Document Embeddings, embeber una respuesta hipotética para mejorar la "
            "búsqueda) y RAG adaptativo que decide dinámicamente cuándo recuperar.\n\n"
            "RAG vs. fine-tuning: RAG es preferible cuando el conocimiento cambia frecuentemente, cuando "
            "se necesitan citas de fuentes, o cuando el dominio es muy específico y los datos de entrenamiento "
            "son escasos. El fine-tuning es mejor cuando se necesita adaptar el estilo, formato o "
            "comportamiento del modelo, o cuando la latencia de recuperación es inaceptable."
        ),
        "fuente": "RAG en Producción - Capítulo 1",
        "fecha": "2023-08-22",
        "version": "1.0",
        "tema": "rag"
    },
    {
        "id": "doc_09",
        "titulo": "Bases de Datos Vectoriales y Búsqueda Aproximada",
        "contenido": (
            "Las bases de datos vectoriales (vector databases) están diseñadas para almacenar, indexar y "
            "buscar eficientemente vectores de alta dimensión, operación central en los sistemas RAG, "
            "motores de recomendación y búsqueda semántica.\n\n"
            "La búsqueda exacta del vecino más cercano (exact kNN) requiere comparar la consulta con "
            "todos los vectores del índice: complejidad O(n·d) donde n es el número de vectores y d la "
            "dimensión. Para millones de documentos esto es inviable en tiempo real. Los algoritmos de "
            "Approximate Nearest Neighbor (ANN) sacrifican exactitud por velocidad mediante indexación.\n\n"
            "HNSW (Hierarchical Navigable Small World) construye un grafo jerárquico de múltiples capas: "
            "las capas superiores tienen pocos nodos (navegación rápida de larga distancia) y las "
            "inferiores tienen todos los nodos (refinamiento local). La búsqueda navega el grafo desde "
            "arriba hacia abajo, alcanzando el vecino aproximado con complejidad O(log n). Es el algoritmo "
            "más usado en producción por su balance precisión-velocidad.\n\n"
            "IVF (Inverted File Index) divide el espacio en celdas de Voronoi mediante k-means y "
            "busca solo en las celdas más cercanas a la consulta. PQ (Product Quantization) comprime "
            "los vectores en códigos compactos para reducir memoria. Los productos como Pinecone, Weaviate, "
            "Qdrant y Milvus implementan estos algoritmos con APIs de alto nivel, soporte para filtros "
            "de metadatos, escalabilidad horizontal y gestión de actualizaciones en tiempo real."
        ),
        "fuente": "Vector Databases - Arquitecturas",
        "fecha": "2023-09-30",
        "version": "1.0",
        "tema": "vector_databases"
    },
    {
        "id": "doc_10",
        "titulo": "Evaluación de Modelos de Lenguaje",
        "contenido": (
            "Evaluar LLMs es sustancialmente más difícil que evaluar modelos tradicionales de ML porque "
            "las salidas son texto libre, subjetivo y multidimensional. Las métricas automáticas clásicas "
            "como BLEU o ROUGE, diseñadas para traducción automática, son insuficientes para capturar "
            "la calidad real de una respuesta.\n\n"
            "Los benchmarks estándar de capacidades generales incluyen MMLU (57 materias académicas, "
            "mide conocimiento factual), HellaSwag (completación de escenas, mide razonamiento de sentido "
            "común), HumanEval (generación de código Python con tests automáticos) y GSM8K (problemas "
            "de matemáticas de primaria que requieren razonamiento en cadena).\n\n"
            "Para sistemas RAG, el framework RAGAS define métricas específicas: Faithfulness (qué "
            "proporción de las afirmaciones de la respuesta están soportadas por el contexto recuperado), "
            "Answer Relevancy (similitud semántica entre la respuesta y la pregunta), Context Precision "
            "(qué fracción del contexto recuperado es relevante para responder) y Context Recall (qué "
            "fracción de la información necesaria está en el contexto recuperado).\n\n"
            "LLM-as-judge usa un LLM (típicamente GPT-4) para evaluar respuestas de otro LLM según "
            "criterios definidos en el prompt. MT-Bench proporciona prompts de evaluación estandarizados "
            "para 8 categorías. Chatbot Arena usa evaluación humana mediante Elo rating, considerada "
            "el gold standard. La calibración con evaluadores humanos es imprescindible para validar "
            "cualquier métrica automática."
        ),
        "fuente": "Evaluación de LLMs - Referencia",
        "fecha": "2023-10-15",
        "version": "1.0",
        "tema": "evaluacion_llm"
    },
    {
        "id": "doc_11",
        "titulo": "Ingeniería de Prompts: Técnicas Avanzadas",
        "contenido": (
            "La ingeniería de prompts es la disciplina de diseñar las instrucciones de entrada a un LLM "
            "para obtener las salidas deseadas. A medida que los modelos se vuelven más capaces, la "
            "calidad del prompt se convierte en el factor determinante del rendimiento en producción.\n\n"
            "Las técnicas fundamentales incluyen: few-shot prompting (proporcionar ejemplos de pares "
            "entrada-salida para que el modelo infiera el patrón), chain-of-thought (pedir razonamiento "
            "paso a paso antes de la respuesta final, mejorando hasta un 40% en benchmarks de matemáticas), "
            "y system prompts que definen el rol, las restricciones y el formato de salida esperado.\n\n"
            "Técnicas avanzadas: Tree of Thoughts permite al modelo explorar múltiples caminos de "
            "razonamiento en paralelo y seleccionar el mejor; ReAct intercala razonamiento y acciones "
            "externas (búsquedas, cálculos) en un bucle interactivo; Self-Consistency genera múltiples "
            "respuestas independientes y selecciona la más frecuente para reducir varianza.\n\n"
            "El meta-prompting usa un LLM para generar o mejorar prompts automáticamente. La inyección "
            "de prompts (prompt injection) es el principal vector de ataque en aplicaciones LLM: un usuario "
            "malintencionado puede insertar instrucciones en el contexto para anular el system prompt. "
            "Las defensas incluyen delimitadores explícitos, validación de entradas y sandboxing. "
            "DSPy ofrece un marco programático para optimizar prompts automáticamente mediante gradientes."
        ),
        "fuente": "Prompt Engineering - Guía Avanzada",
        "fecha": "2023-11-01",
        "version": "1.0",
        "tema": "prompt_engineering"
    },
    {
        "id": "doc_12",
        "titulo": "Embeddings Semánticos: Modelos y Métricas",
        "contenido": (
            "Los embeddings semánticos son representaciones vectoriales densas del texto donde la "
            "proximidad geométrica refleja similitud semántica. Son la tecnología central de los sistemas "
            "de búsqueda semántica, sistemas RAG y clasificación sin etiquetas.\n\n"
            "Los modelos de embedding más utilizados en la actualidad son: text-embedding-3-small y "
            "text-embedding-3-large de OpenAI (1536 y 3072 dimensiones respectivamente), con excelente "
            "rendimiento en benchmarks MTEB; E5-large de Microsoft, entrenado con pares de preguntas y "
            "párrafos de respuesta; y BGE de BAAI, especialmente fuerte en recuperación en chino e inglés. "
            "INSTRUCTOR permite especificar la tarea en el prompt del modelo de embedding para obtener "
            "representaciones óptimas para cada caso de uso.\n\n"
            "Las métricas de similitud más comunes son: similitud coseno (independiente de la magnitud del "
            "vector, recomendada para la mayoría de tareas), producto escalar (equivalente al coseno si "
            "los vectores están normalizados), y distancia euclidiana (L2). Beir Benchmark y MTEB "
            "(Massive Text Embedding Benchmark) evalúan embeddings en decenas de tareas y datasets.\n\n"
            "La dimensionalidad tiene un trade-off: más dimensiones capturan más matices semánticos pero "
            "aumentan el coste de almacenamiento y cómputo. La técnica Matryoshka Representation Learning "
            "entrena embeddings que mantienen calidad al truncarse, permitiendo elegir dimensionalidad "
            "en tiempo de inferencia. Los embeddings multimodales (CLIP, ImageBind) proyectan imágenes "
            "y texto al mismo espacio vectorial, habilitando búsqueda cross-modal."
        ),
        "fuente": "Embeddings y Búsqueda Semántica",
        "fecha": "2023-12-10",
        "version": "1.0",
        "tema": "embeddings"
    },
    {
        "id": "doc_13",
        "titulo": "Redes Generativas Adversariales (GANs)",
        "contenido": (
            "Las Redes Generativas Adversariales (GANs), propuestas por Ian Goodfellow en 2014, son un "
            "marco de entrenamiento donde dos redes neuronales compiten entre sí en un juego minimax: "
            "el generador G intenta crear muestras que engañen al discriminador D, mientras que D intenta "
            "distinguir muestras reales de generadas. El entrenamiento converge (en teoría) cuando G "
            "produce una distribución idéntica a la de los datos reales.\n\n"
            "La función objetivo es: min_G max_D E[log D(x)] + E[log(1 - D(G(z)))], donde x son muestras "
            "reales y z es ruido latente. En la práctica, el entrenamiento de GANs es notoriamente "
            "inestable: el colapso de modo (mode collapse) hace que G solo genere un subconjunto de la "
            "distribución; la oscilación impide que ambas redes converjan simultáneamente.\n\n"
            "Las mejoras arquitectónicas han mitigado estos problemas: DCGAN introdujo convoluciones "
            "transpuestas y Batch Normalization para estabilizar el entrenamiento de imágenes. "
            "Wasserstein GAN (WGAN) reemplaza la divergencia Jensen-Shannon por la distancia de "
            "Wasserstein, proporcionando gradientes más informativos. StyleGAN y StyleGAN2 separaron "
            "el control del estilo y el contenido a través de un mapeo de espacio latente, logrando "
            "imágenes de rostros fotorrealistas (proyecto ThisPersonDoesNotExist).\n\n"
            "Las aplicaciones van desde síntesis de imágenes y vídeos hasta aumento de datos, "
            "super-resolución (SRGAN), transferencia de estilo (CycleGAN para traducción imagen a imagen "
            "sin pares supervisados) y generación de datos sintéticos para protección de privacidad. "
            "Los modelos de difusión han superado a las GANs en calidad y diversidad en muchas tareas."
        ),
        "fuente": "Modelos Generativos - Módulo 2",
        "fecha": "2024-01-25",
        "version": "1.0",
        "tema": "gans"
    },
    {
        "id": "doc_14",
        "titulo": "Modelos de Difusión: DDPM y Stable Diffusion",
        "contenido": (
            "Los modelos de difusión son modelos generativos que aprenden a revertir un proceso de "
            "destrucción gradual de información. En el proceso forward, se añade ruido gaussiano "
            "progresivamente a una imagen durante T pasos hasta obtener ruido puro. El modelo aprende "
            "el proceso inverso: dado el resultado ruidoso en el paso t, predice el ruido añadido para "
            "poder reconstruir la imagen en el paso t-1.\n\n"
            "DDPM (Denoising Diffusion Probabilistic Models, Ho et al. 2020) formalizó este proceso: "
            "una U-Net con atención aprende a predecir el ruido ε añadido en cada paso. La función de "
            "pérdida es simplemente el MSE entre el ruido real y el predicho. El muestreo requiere T "
            "pasos iterativos (T=1000 en DDPM original), lo que hace la generación lenta. DDIM "
            "(Denoising Diffusion Implicit Models) reduce T a 50-100 pasos sin reentrenamiento.\n\n"
            "Stable Diffusion opera en el espacio latente de un autoencoder variacional (VAE): el VAE "
            "comprime la imagen a una representación 8x más pequeña, la difusión opera en ese espacio "
            "latente (mucho más eficiente), y el VAE decodifica el resultado. La condición de texto se "
            "inyecta mediante cross-attention con embeddings de CLIP. Esto reduce la memoria necesaria "
            "de decenas de GB a menos de 8 GB.\n\n"
            "Las técnicas de control incluyen: CFG (Classifier-Free Guidance) que interpola entre "
            "generación condicional e incondicional para controlar la adherencia al prompt; ControlNet "
            "añade condicionamiento espacial (bordes Canny, poses humanas, profundidad) sin reentrenar "
            "el modelo base; y LoRA para fine-tuning eficiente de estilos artísticos. DALL-E 3, "
            "Midjourney v6 e Imagen 3 son las implementaciones comerciales más avanzadas del paradigma."
        ),
        "fuente": "Modelos de Difusión - Referencia",
        "fecha": "2024-02-14",
        "version": "1.0",
        "tema": "difusion"
    },
    {
        "id": "doc_15",
        "titulo": "Ética en Inteligencia Artificial: Riesgos y Salvaguardas",
        "contenido": (
            "La ética en IA aborda los principios, valores y salvaguardas necesarios para que los "
            "sistemas de inteligencia artificial sean seguros, justos, transparentes y beneficiosos "
            "para la sociedad. A medida que los sistemas de IA se despliegan en dominios críticos "
            "(salud, justicia, crédito, contratación), las implicaciones de sus fallos se multiplican.\n\n"
            "El sesgo algorítmico surge cuando los modelos aprenden y amplifican sesgos presentes en "
            "los datos de entrenamiento. El caso más documentado es COMPAS, un sistema de predicción "
            "de reincidencia criminal que clasificaba erróneamente a personas de color como de alto "
            "riesgo el doble de veces que a personas blancas. Los sesgos pueden ser de representación "
            "(subgrupos infrarrepresentados en el entrenamiento) o de medición (etiquetas históricamente "
            "sesgadas).\n\n"
            "La explicabilidad (XAI, Explainable AI) busca hacer comprensibles las decisiones de "
            "modelos complejos. LIME genera explicaciones locales aproximando el modelo con uno "
            "interpretable en el entorno de la instancia a explicar. SHAP (Shapley Additive exPlanations) "
            "asigna a cada característica su contribución marginal promedio usando teoría de juegos "
            "cooperativos. Los modelos de atención en transformers ofrecen interpretabilidad parcial, "
            "aunque la investigación muestra que la atención no siempre refleja causalidad.\n\n"
            "Los riesgos existenciales y de alineación incluyen: el problema de especificación de "
            "objetivos (reward hacking), la distribución shift entre entrenamiento y despliegue, y los "
            "riesgos de sistemas autónomos con objetivos mal definidos. Los marcos regulatorios emergentes "
            "como el AI Act de la UE establecen niveles de riesgo y requisitos de transparencia. El "
            "movimiento de IA responsable aboga por red-teaming, auditorías externas y participación "
            "ciudadana en el diseño de sistemas de alto impacto."
        ),
        "fuente": "Ética en IA - Marco Regulatorio",
        "fecha": "2024-03-08",
        "version": "1.0",
        "tema": "etica_ia"
    }
]

print(f"✓ Corpus cargado: {len(documentos)} documentos")
for doc in documentos:
    print(f"  [{doc['id']}] {doc['titulo']} ({doc['tema']})")


El corpus cubre las 15 áreas temáticas más relevantes del máster. Cada documento tiene contenido técnico genuino de 200-350 palabras que el sistema RAG podrá recuperar y citar. Los metadatos (`fuente`, `fecha`, `version`, `tema`) permitirán demostrar el filtrado semántico en la Sección 9.

**Celda 5: Tabla resumen del corpus**

In [ ]:
# Construir DataFrame resumen
filas_resumen = []
for doc in documentos:
    filas_resumen.append({
        "id": doc["id"],
        "titulo": doc["titulo"][:45] + "..." if len(doc["titulo"]) > 45 else doc["titulo"],
        "tema": doc["tema"],
        "fecha": doc["fecha"],
        "version": doc["version"],
        "palabras": len(doc["contenido"].split()),
        "chars": len(doc["contenido"])
    })

df_corpus = pd.DataFrame(filas_resumen)
print("=== TABLA RESUMEN DEL CORPUS ===")
print(df_corpus.to_string(index=False))
print(f"\nTotal de palabras en el corpus: {df_corpus['palabras'].sum():,}")
print(f"Promedio de palabras por documento: {df_corpus['palabras'].mean():.0f}")
print(f"Mín/Máx palabras: {df_corpus['palabras'].min()} / {df_corpus['palabras'].max()}")

# Mostrar el primer documento completo
print("\n" + "="*60)
print(f"DOCUMENTO COMPLETO: {documentos[0]['titulo']}")
print("="*60)
print(f"ID      : {documentos[0]['id']}")
print(f"Fuente  : {documentos[0]['fuente']}")
print(f"Fecha   : {documentos[0]['fecha']}")
print(f"Versión : {documentos[0]['version']}")
print(f"Tema    : {documentos[0]['tema']}")
print(f"\nContenido:\n{documentos[0]['contenido']}")


El DataFrame resume las propiedades clave del corpus. Observa que todos los documentos tienen entre 250 y 350 palabras: suficiente contenido para que el chunking produzca 2-3 fragmentos por documento con solapamiento, generando un índice de aproximadamente 30-40 chunks.

## Sección 3: Limpieza y Chunking

**Celda 6: Función de limpieza de texto**

In [ ]:
import unicodedata

def limpiar_texto(texto: str) -> str:
    """
    Limpia y normaliza un texto para indexación:
    1. Normaliza unicode a NFC (compone caracteres compuestos)
    2. Elimina caracteres de control (excepto \n y \t)
    3. Colapsa espacios múltiples en uno solo
    4. Elimina líneas en blanco múltiples consecutivas
    5. Elimina espacios al inicio y final
    """
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###

# Prueba de la función
texto_sucio = "  Machine   learning\x00 es\x01 una\n\n\n\nsubdisciplina  de  la IA.  "
texto_limpio = limpiar_texto(texto_sucio)
print(f"Original : {repr(texto_sucio)}")
print(f"Limpio   : {repr(texto_limpio)}")

# Aplicar limpieza a todos los documentos
for doc in documentos:
    doc["contenido"] = limpiar_texto(doc["contenido"])

print(f"\n✓ Limpieza aplicada a {len(documentos)} documentos.")


La limpieza de texto es el primer paso del pipeline ETL en un sistema RAG. Los problemas más frecuentes en texto real son: caracteres de control incrustados (BOM, null bytes), codificación inconsistente (e.g. ñ representada como secuencias de bytes distintas según el sistema fuente), y espacios múltiples que inflan el conteo de tokens sin aportar información. La normalización NFC es especialmente importante en español para que "á" sea siempre un único carácter, no dos (a + acento combinante).

**Celda 7: Chunking por tokens con solapamiento**

In [ ]:
def chunking_por_tokens(texto: str, max_tokens: int = 150,
                         solapamiento: int = 30,
                         encoding_name: str = "cl100k_base") -> list:
    """
    Divide un texto en fragmentos (chunks) de tamaño máximo max_tokens tokens,
    con solapamiento de solapamiento tokens entre chunks consecutivos.

    Args:
        texto: Texto de entrada (ya limpiado).
        max_tokens: Máximo de tokens por chunk (default 150).
        solapamiento: Tokens de solapamiento entre chunks (default 30).
        encoding_name: Nombre del encoding de tiktoken (cl100k_base para GPT-4).

    Returns:
        Lista de strings, cada uno con max_tokens tokens como máximo.
    """
    enc = tiktoken.get_encoding(encoding_name)
    tokens = enc.encode(texto)
    chunks = []
    # Pista: paso = max_tokens - solapamiento
    # Usa un bucle while: extrae tokens[inicio:fin], decodifica, añade a chunks
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Aplicar chunking a todos los documentos
chunks_totales = []
chunk_global_id = 0

for doc in documentos:
    fragmentos = chunking_por_tokens(doc["contenido"], max_tokens=150, solapamiento=30)
    for idx, fragmento in enumerate(fragmentos):
        chunks_totales.append({
            "chunk_id": f"chunk_{chunk_global_id:04d}",
            "texto": fragmento,
            "doc_id": doc["id"],
            "fuente": doc["fuente"],
            "fecha": doc["fecha"],
            "version": doc["version"],
            "tema": doc["tema"],
            "titulo": doc["titulo"],
            "chunk_index": idx,
            "total_chunks": len(fragmentos)
        })
        chunk_global_id += 1

print(f"✓ Chunking completado.")
print(f"  Documentos procesados : {len(documentos)}")
print(f"  Chunks generados      : {len(chunks_totales)}")
print(f"  Promedio chunks/doc   : {len(chunks_totales)/len(documentos):.1f}")


El chunking por tokens es superior al chunking por caracteres o palabras porque respeta los límites reales del modelo de embedding. Usar el mismo encoding (`cl100k_base`) que los modelos de OpenAI garantiza que cada chunk nunca exceda la ventana de contexto del modelo de embeddings. El solapamiento de 30 tokens evita que el boundary entre chunks corte frases en mitad de una idea: los tokens del final del chunk N reaparecen al inicio del chunk N+1, preservando la coherencia semántica en los bordes.

**Celda 8: Estadísticas de los chunks**

In [ ]:
enc = tiktoken.get_encoding("cl100k_base")

# Calcular tokens por chunk
tokens_por_chunk = [len(enc.encode(c["texto"])) for c in chunks_totales]
chars_por_chunk = [len(c["texto"]) for c in chunks_totales]

print("=== ESTADÍSTICAS DE CHUNKING ===")
print(f"Total de chunks         : {len(chunks_totales)}")
print(f"Tokens por chunk:")
print(f"  Mínimo                : {min(tokens_por_chunk)}")
print(f"  Máximo                : {max(tokens_por_chunk)}")
print(f"  Promedio              : {sum(tokens_por_chunk)/len(tokens_por_chunk):.1f}")
print(f"  Mediana               : {sorted(tokens_por_chunk)[len(tokens_por_chunk)//2]}")
print(f"Total tokens en corpus  : {sum(tokens_por_chunk):,}")

# Mostrar 3 chunks de ejemplo
print("\n=== 3 CHUNKS DE EJEMPLO ===")
for i in [0, len(chunks_totales)//2, -1]:
    c = chunks_totales[i]
    n_tokens = len(enc.encode(c["texto"]))
    print(f"\n{'─'*55}")
    print(f"chunk_id    : {c['chunk_id']}")
    print(f"doc_id      : {c['doc_id']}  ({c['titulo'][:40]}...)"
          if len(c["titulo"]) > 40 else f"doc_id      : {c['doc_id']}  ({c['titulo']})")
    print(f"chunk_index : {c['chunk_index']+1}/{c['total_chunks']}")
    print(f"tema        : {c['tema']}")
    print(f"fuente      : {c['fuente']}")
    print(f"tokens      : {n_tokens}")
    print(f"texto       : {c['texto'][:200]}{'...' if len(c['texto'])>200 else ''}")


Las estadísticas confirman que la mayoría de chunks tienen entre 120 y 150 tokens (los últimos chunks de cada documento pueden ser más cortos). El solapamiento de 30 tokens representa aproximadamente el 20% de cada chunk, un valor que empíricamente equilibra bien la preservación de contexto en los bordes con la redundancia de información.

## Sección 4: Metadatos e Índice en ChromaDB

**Celda 9: Función de embedding e inicialización de ChromaDB**

In [ ]:
def generar_embedding(texto: str) -> list:
    """
    Genera el embedding de un texto usando text-embedding-3-small de OpenAI.
    Devuelve una lista de 1536 floats.
    Usa: client.embeddings.create(model="text-embedding-3-small", input=[texto])
    Accede al embedding con: resp.data[0].embedding
    """
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Inicializar ChromaDB en modo efímero (en memoria, sin persistencia en disco)
chroma_client = chromadb.EphemeralClient()
try:
    chroma_client.delete_collection("rag_avanzado")
except Exception:
    pass
collection = chroma_client.create_collection(
    name="rag_avanzado",
    metadata={"hnsw:space": "cosine"}
)
print("✓ ChromaDB inicializado (modo efímero).")

emb_prueba = generar_embedding("prueba de embedding")
print(f"✓ Embedding generado: {len(emb_prueba)} dimensiones")
print(f"  Primeros 5 valores: {[round(v, 4) for v in emb_prueba[:5]]}")


Usamos `EphemeralClient()` (en memoria) en lugar del cliente persistente para el entorno de laboratorio: no requiere configurar directorios y se resetea limpiamente al reiniciar el kernel. En producción se usaría `chromadb.PersistentClient(path="./chroma_db")` para mantener el índice entre sesiones. La métrica coseno es la estándar para embeddings de texto: independiente de la magnitud del vector, mide exclusivamente la orientación angular.

**Celda 10: Indexación de todos los chunks en ChromaDB**

In [ ]:
BATCH_SIZE = 50

print(f"Indexando {len(chunks_totales)} chunks en ChromaDB (lotes de {BATCH_SIZE})...")
print("─" * 55)

inicio_total = time.time()

for i in range(0, len(chunks_totales), BATCH_SIZE):
    lote = chunks_totales[i : i + BATCH_SIZE]

    ids = [c["chunk_id"] for c in lote]
    textos = [c["texto"] for c in lote]
    metadatas = [
        {
            "doc_id": c["doc_id"],
            "titulo": c["titulo"],
            "fuente": c["fuente"],
            "fecha": c["fecha"],
            "version": c["version"],
            "tema": c["tema"],
            "chunk_index": c["chunk_index"],
            "total_chunks": c["total_chunks"]
        }
        for c in lote
    ]

    # Generar embeddings para este lote
    embeddings = [generar_embedding(t) for t in textos]

    # Añadir a ChromaDB
    collection.add(
        documents=textos,
        embeddings=embeddings,
        metadatas=metadatas,
        ids=ids
    )

    fin_lote = i + len(lote)
    print(f"  Lote {i//BATCH_SIZE + 1}: chunks {i+1}–{fin_lote} indexados ✓")

tiempo_total = time.time() - inicio_total
total_indexados = collection.count()

print(f"\n{'='*55}")
print(f"✓ Indexación completada en {tiempo_total:.1f}s")
print(f"  Total de chunks en la colección: {total_indexados}")
print(f"  Coste estimado de embeddings   : ${total_indexados * 0.02 / 1_000_000:.6f} USD")
print(f"  (text-embedding-3-small: $0.02 / 1M tokens)")


El procesamiento por lotes (batches) reduce el overhead de llamadas a la API. Con 50 chunks por lote sobre ~40 chunks totales, prácticamente todo se procesa en un único lote. En producción con millones de documentos, la paralelización y el batching son críticos para la eficiencia. Nota el coste mínimo: text-embedding-3-small es extremadamente económico ($0.02 por millón de tokens), lo que lo hace viable para corpus de gran escala.

**Celda 11: Verificación del índice**

In [ ]:
# Verificar el índice con una pregunta de prueba
pregunta_test = "¿Qué son los transformers?"

# Generar embedding de la pregunta
emb_pregunta = generar_embedding(pregunta_test)

# Consultar ChromaDB
resultados_test = collection.query(
    query_embeddings=[emb_pregunta],
    n_results=3,
    include=["documents", "metadatas", "distances"]
)

print(f"=== VERIFICACIÓN DEL ÍNDICE ===")
print(f"Pregunta: '{pregunta_test}'")
print(f"Top-3 resultados:\n")

for rank, (doc, meta, dist) in enumerate(zip(
    resultados_test["documents"][0],
    resultados_test["metadatas"][0],
    resultados_test["distances"][0]
), start=1):
    similitud = 1 - dist   # ChromaDB con métrica coseno devuelve distancia, no similitud
    print(f"  Resultado #{rank}")
    print(f"    Fuente   : {meta['fuente']}")
    print(f"    Fecha    : {meta['fecha']}")
    print(f"    Tema     : {meta['tema']}")
    print(f"    Chunk    : {meta['chunk_index']+1}/{meta['total_chunks']}")
    print(f"    Distancia: {dist:.4f}  |  Similitud: {similitud:.4f}")
    print(f"    Texto    : {doc[:150]}...")
    print()

print(f"Total de vectores en el índice: {collection.count()}")


La verificación confirma que el índice funciona correctamente. Los tres resultados deberían pertenecer al documento sobre Transformers (`doc_06`). La similitud coseno de 0.85+ indica alta relevancia semántica. Nótese la conversión `similitud = 1 - distancia`: ChromaDB con métrica coseno devuelve la distancia (0=idéntico, 2=opuesto), no la similitud.

## Sección 5: Retrieval Híbrido

**Celda 12: Búsqueda semántica**

In [ ]:
def busqueda_semantica(pregunta: str, n: int = 5) -> list:
    """
    Busca los n fragmentos más similares semánticamente a la pregunta.
    1. Genera el embedding de la pregunta con generar_embedding()
    2. Llama a collection.query(query_embeddings=[emb], n_results=n,
       include=["documents","metadatas","distances"])
    3. Construye y devuelve una lista de dicts con los campos:
       chunk_id, texto, titulo, fuente, fecha, tema, version,
       chunk_index, distancia, similitud (= 1 - distancia)
    """
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Prueba
pregunta_demo = "¿Cómo funciona el mecanismo de atención?"
resultados_sem = busqueda_semantica(pregunta_demo, n=5)

print(f"=== BÚSQUEDA SEMÁNTICA ===")
print(f"Pregunta: '{pregunta_demo}'\n")
for i, r in enumerate(resultados_sem, 1):
    print(f"  #{i} [{r['chunk_id']}] {r['titulo'][:50]}")
    print(f"     Tema: {r['tema']} | Similitud: {r['similitud']:.4f}")
    print(f"     Texto: {r['texto'][:120]}...")
    print()


La búsqueda semántica convierte la pregunta a un vector de 1536 dimensiones y encuentra los fragmentos más cercanos en el espacio de embeddings. Es excelente para capturar paráfrasis y sinónimos: "mecanismo de atención" recuperará textos que hablen de "attention mechanism" o "self-attention" aunque no usen exactamente esas palabras. Su debilidad es que puede fallar con términos técnicos específicos (siglas, nombres propios, números) que no tienen vecinos semánticos claros.

**Celda 13: Búsqueda BM25**

In [ ]:
# Construir índice BM25 sobre todos los chunks
textos_chunks = [c["texto"] for c in chunks_totales]

def tokenizar_bm25(texto: str) -> list:
    texto = texto.lower()
    return re.findall(r"[a-záéíóúüñ]+", texto)

corpus_tokenizado = [tokenizar_bm25(t) for t in textos_chunks]
bm25_index = BM25Okapi(corpus_tokenizado)
print(f"✓ Índice BM25 construido sobre {len(corpus_tokenizado)} chunks.")


def busqueda_bm25(pregunta: str, n: int = 5) -> list:
    """
    Busca los n fragmentos más relevantes usando BM25Okapi.
    1. Tokeniza la pregunta con tokenizar_bm25()
    2. Obtén scores con bm25_index.get_scores(tokens_query)
    3. Ordena índices descendente: np.argsort(scores)[::-1][:n]
    4. Devuelve lista de dicts con: chunk_id, texto, titulo, fuente,
       fecha, tema, version, chunk_index, score_bm25
    """
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Prueba con la misma pregunta
resultados_bm25 = busqueda_bm25(pregunta_demo, n=5)
print(f"\n=== BÚSQUEDA BM25 ===")
print(f"Pregunta: '{pregunta_demo}'\n")
for i, r in enumerate(resultados_bm25, 1):
    print(f"  #{i} [{r['chunk_id']}] {r['titulo'][:50]}")
    print(f"     Tema: {r['tema']} | Score BM25: {r['score_bm25']:.4f}")
    print(f"     Texto: {r['texto'][:120]}...")
    print()


BM25 (Best Match 25) es un modelo probabilístico de recuperación de información basado en TF-IDF con saturación de frecuencia. La fórmula de BM25Okapi pondera cada término por: (1) su frecuencia en el documento (TF), saturada por el parámetro k1 para evitar que términos muy frecuentes dominen; (2) la rareza del término en el corpus (IDF); y (3) normalización por la longitud del documento. BM25 es excelente para términos técnicos exactos: "atención", "transformer", "softmax" serán recuperados precisamente. Su debilidad es no entender sinónimos ni paráfrasis.

## Sección 6: Re-ranking con Cross-Encoder

**Celda 16: Carga del Cross-Encoder y función de re-ranking**

In [ ]:
# Cargar el Cross-Encoder para re-ranking
print("Cargando Cross-Encoder (puede tardar la primera vez)...")
cross_encoder = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"✓ Cross-Encoder cargado")


def reranking(pregunta: str, candidatos: list, top_k: int = 3) -> list:
    """
    Re-rankea chunks candidatos usando un Cross-Encoder.
    1. Crea pares: [(pregunta, c['texto']) for c in candidatos]
    2. Puntúa con: scores = cross_encoder.predict(pares).tolist()
    3. Añade 'score_crossencoder' a cada candidato
    4. Ordena por score descendente y devuelve los top_k
    """
    if not candidatos:
        return []
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Prueba rápida
print("\nProbando el Cross-Encoder...")
s1 = cross_encoder.predict([("¿Qué es un transformer?",
    "Los transformers usan mecanismos de atención para procesar secuencias.")])
s2 = cross_encoder.predict([("¿Qué es un transformer?",
    "La gastronomía española incluye la paella y el gazpacho.")])
print(f"Score relevante  : {s1[0]:.4f}")
print(f"Score irrelevante: {s2[0]:.4f}")


El Cross-Encoder es un modelo transformer que recibe el par completo (pregunta, fragmento) y produce un único score de relevancia. La clave de su superioridad sobre los bi-encoders está en la **interacción cruzada**: los tokens de la pregunta pueden atender directamente a los tokens del fragmento en todas las capas del transformer, capturando matices sutiles de relevancia. El precio es la escala: con N candidatos hay que hacer N inferencias, por lo que se usa únicamente para re-rankear los top-20 candidatos del retrieval, no el corpus completo.

**Celda 17: Aplicar re-ranking**


In [ ]:
# Aplicar re-ranking a los resultados de búsqueda semántica
chunks_rerankeados = reranking(pregunta_demo, resultados_sem[:7], top_k=3)

print(f"=== COMPARATIVA ANTES/DESPUÉS DEL RE-RANKING ===")
print(f"Pregunta: '{pregunta_demo}'\n")

print("── ANTES (ranking semántico) ──")
filas_antes = []
for rank, r in enumerate(resultados_sem[:7], 1):
    filas_antes.append({
        "Rank Semántico": rank,
        "chunk_id": r["chunk_id"],
        "Título": r["titulo"][:40] + "..." if len(r["titulo"]) > 40 else r["titulo"],
        "Similitud": round(r["similitud"], 4)
    })
import pandas as pd
print(pd.DataFrame(filas_antes).to_string(index=False))

print("\n── DESPUÉS (re-ranking Cross-Encoder) ──")
filas_despues = []
for rank, r in enumerate(chunks_rerankeados, 1):
    rank_orig = next((j+1 for j, s in enumerate(resultados_sem[:7]) if s["chunk_id"] == r["chunk_id"]), "?")
    filas_despues.append({
        "Rank CE": rank,
        "Rank ant.": rank_orig,
        "chunk_id": r["chunk_id"],
        "Título": r["titulo"][:40] + "..." if len(r["titulo"]) > 40 else r["titulo"],
        "Score CE": round(r["score_ce"], 4)
    })
print(pd.DataFrame(filas_despues).to_string(index=False))


La tabla muestra el impacto del re-ranking: chunks que estaban en posiciones intermedias del ranking RRF pueden ascender a los primeros puestos si el Cross-Encoder determina que tienen mayor relevancia para la pregunta concreta. Esto es especialmente visible cuando la búsqueda semántica recupera textos temáticamente relacionados pero el Cross-Encoder distingue cuál responde *directamente* a la pregunta.

**Celda 18: Visualización textual del cambio de ranking**

In [ ]:
print("=" * 60)
print("ANTES del re-ranking (Top-7 semántico)")
print("=" * 60)
for rank, r in enumerate(resultados_sem[:7], 1):
    barra = "█" * min(int(r["similitud"] * 30), 30)
    print(f"  #{rank} {barra} | {r['chunk_id']} | {r['titulo'][:40]}")
    print(f"       Similitud: {r['similitud']:.4f}")

print()
print("=" * 60)
print("DESPUÉS del re-ranking (Top-3 por Cross-Encoder)")
print("=" * 60)
for rank, r in enumerate(chunks_rerankeados, 1):
    rank_orig = next((j+1 for j, s in enumerate(resultados_sem[:7]) if s["chunk_id"] == r["chunk_id"]), "?")
    cambio = f"#{rank_orig} → #{rank}"
    print(f"  #{rank} ({cambio}) | {r['chunk_id']} | {r['titulo'][:40]}")
    print(f"       Score CE: {r['score_ce']:.4f}")


La visualización hace evidente el efecto del re-ranking. Los scores del Cross-Encoder son logits no normalizados (no son probabilidades): un score de 5.0 es muy relevante, 0.0 es neutro y valores muy negativos (-10 o menos) indican irrelevancia. El pipeline completo de retrieval sigue la arquitectura estándar en producción: candidatos amplios (top-20 o top-50) mediante métodos rápidos → re-ranking preciso de candidatos finalistas (top-3 o top-5) con el Cross-Encoder.

## Sección 7: Generación con Grounding y Citas

**Celda 19: Construcción del contexto con formato de citas**

In [ ]:
def construir_contexto(chunks_rerankeados: list) -> str:
    """
    Formatea los chunks como bloque de contexto con citas numeradas [N].

    Para cada chunk (enumerado desde 1), añade:
    '[N] {texto del chunk}'
    '   📌 Fuente: {fuente} | Fecha: {fecha} | Tema: {tema}'
    Envuelve todo con cabecera 'CONTEXTO RECUPERADO:' y separadores '─'*50

    Devuelve el string completo.
    """
    lineas = ["CONTEXTO RECUPERADO:"]
    lineas.append("─" * 50)
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Mostrar el contexto para los chunks re-rankeados
chunks_rerankeados = reranking(pregunta_demo, resultados_rrf, top_k=3)
contexto_ejemplo = construir_contexto(chunks_rerankeados)
print(contexto_ejemplo)

enc = tiktoken.get_encoding("cl100k_base")
tokens_ctx = len(enc.encode(contexto_ejemplo))
print(f"\nTokens en el contexto: {tokens_ctx}")
print(f"Coste estimado (input): ${tokens_ctx * 0.15 / 1_000_000:.6f} USD")


El formato `[N]` para las citas es una convención que el LLM entiende naturalmente gracias a su entrenamiento en texto académico y web. Al incluir la numeración explícita en el contexto, el prompt del sistema puede instruir al modelo a citar usando `[1]`, `[2]`, `[3]`, creando respuestas verificables y transparentes. Las estadísticas de tokens son importantes: el contexto de 3 chunks ocupa generalmente 400-500 tokens, una fracción pequeña de la ventana de 128K tokens de gpt-4o-mini.

**Celda 20: Función de generación RAG con grounding**

In [ ]:
def responder_con_citas(pregunta: str, chunks_rerankeados: list) -> str:
    """
    Genera una respuesta fundamentada en los chunks recuperados usando el LLM.
    El prompt de sistema instruye al modelo a:
    1. Usar ÚNICAMENTE la información del contexto proporcionado.
    2. Citar cada afirmación con [N] referenciando el chunk correspondiente.
    3. Admitir explícitamente si no tiene información suficiente.
    4. No inventar datos ni extrapolar más allá del contexto.

    Args:
        pregunta: Pregunta del usuario.
        chunks_rerankeados: Lista de chunks recuperados y re-rankeados.

    Returns:
        Texto de la respuesta generada con citas.
    """
    contexto = construir_contexto(chunks_rerankeados)

    system_prompt = (
        "Eres un asistente experto en Inteligencia Artificial y Machine Learning. "
        "Tu misión es responder preguntas basándote ÚNICAMENTE en el CONTEXTO proporcionado.\n\n"
        "REGLAS ESTRICTAS:\n"
        "1. Para cada afirmación importante, cita la fuente usando [N] donde N es el número "
        "del fragmento del contexto.\n"
        "2. Si el contexto no contiene información suficiente para responder, di exactamente: "
        "'No tengo información sobre esto en los documentos disponibles.'\n"
        "3. NO inventes datos, fechas, cifras ni nombres que no aparezcan en el contexto.\n"
        "4. NO extrapoques ni añadas conocimiento propio más allá de lo que dice el contexto.\n"
        "5. Estructura tu respuesta con claridad: responde directamente y luego añade matices si procede."
    )

    user_message = (
        f"{contexto}\n\n"
        f"PREGUNTA: {pregunta}\n\n"
        "Por favor, responde basándote exclusivamente en el contexto anterior, citando las fuentes con [N]."
    )

    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )

    return response.output_text


# Prueba con los chunks ya rerankeados
print(f"=== GENERACIÓN CON GROUNDING ===")
print(f"Pregunta: '{pregunta_demo}'\n")
respuesta_demo = responder_con_citas(pregunta_demo, chunks_rerankeados)
print(respuesta_demo)


El diseño del system prompt es la clave para un RAG confiable. Las cinco reglas son complementarias: la regla 1 fuerza las citas; la regla 2 establece la respuesta de fallback cuando no hay información; las reglas 3 y 4 prohíben explícitamente la alucinación; la regla 5 guía el formato de la respuesta. En producción, se recomienda incluir también la fecha actual en el system prompt para que el modelo no confunda conocimiento de su preentrenamiento con la información del contexto.

**Celda 21: Demo end-to-end con 3 preguntas**

In [ ]:
def pipeline_rag_completo(pregunta: str, n_candidatos: int = 7, top_k: int = 3) -> dict:
    """
    Pipeline RAG: búsqueda semántica + re-ranking + generación.
    """
    t0 = time.time()

    res_sem = busqueda_semantica(pregunta, n=n_candidatos)
    chunks_finales = reranking(pregunta, res_sem[:n_candidatos], top_k=top_k)
    respuesta = responder_con_citas(pregunta, chunks_finales)

    return {
        "pregunta": pregunta,
        "respuesta": respuesta,
        "chunks_finales": chunks_finales,
        "latencia": round(time.time() - t0, 2),
        "n_sem": len(res_sem),
        "n_rerankeados": len(chunks_finales)
    }


preguntas_demo = [
    "¿Cómo funciona el mecanismo de atención en los Transformers?",
    "¿Cuáles son los riesgos éticos de la IA?",
    "¿Qué diferencia hay entre fine-tuning y RAG?"
]

sep = "=" * 65
for i, pregunta in enumerate(preguntas_demo, 1):
    print(sep)
    print(f"PREGUNTA {i}: {pregunta}")
    print(sep)

    resultado = pipeline_rag_completo(pregunta)

    print("\nFuentes recuperadas:")
    for j, c in enumerate(resultado["chunks_finales"], 1):
        print(f"  [{j}] {c['titulo']} ({c['fuente']}, {c['fecha']})")

    print("\nRESPUESTA:")
    print(resultado["respuesta"])
    print(f"\nLatencia: {resultado['latencia']}s")
    print()


El pipeline end-to-end encadena las cinco etapas en una función reutilizable. Observa que cada pregunta activa caminos distintos de recuperación: la pregunta sobre Transformers recupera chunks de `doc_06`; la de ética recupera de `doc_15`; la comparativa fine-tuning vs. RAG combina `doc_07` y `doc_08`. Esta diversidad demuestra que el índice funciona correctamente como memoria factual del sistema.

## Sección 8: Evaluación del Sistema RAG

**Celda 22: Relevancia de recuperación por similitud coseno**

In [ ]:
def cosine_sim(a: list, b: list) -> float:
    """Similitud coseno entre dos vectores."""
    a_np = np.array(a, dtype=float)
    b_np = np.array(b, dtype=float)
    norma_a = np.linalg.norm(a_np)
    norma_b = np.linalg.norm(b_np)
    if norma_a == 0 or norma_b == 0:
        return 0.0
    return float(np.dot(a_np, b_np) / (norma_a * norma_b))


def evaluar_relevancia(pregunta: str, chunks_recuperados: list) -> dict:
    """
    Evalúa relevancia midiendo similitud coseno entre embedding de
    la pregunta y embedding de cada chunk.

    Para cada chunk:
    1. Genera su embedding con generar_embedding(c['texto'])
    2. Calcula cosine_sim(emb_pregunta, emb_chunk)
    3. Guarda en lista scores y lista detalles
    Devuelve dict con: relevancia_media, scores_por_chunk, detalles
    """
    emb_pregunta = generar_embedding(pregunta)
    scores = []
    detalles = []
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Evaluar la relevancia del pipeline demo
pregunta_eval = "¿Cómo funciona el mecanismo de atención en los Transformers?"
res_eval = pipeline_rag_completo(pregunta_eval)
metricas_rel = evaluar_relevancia(pregunta_eval, res_eval["chunks_finales"])

print(f"=== EVALUACIÓN DE RELEVANCIA ===")
print(f"Pregunta: '{pregunta_eval}'\n")
print(f"Relevancia media: {metricas_rel['relevancia_media']:.4f}\n")
for det in metricas_rel["detalles"]:
    barra = "█" * int(det["similitud"] * 40)
    print(f"  {barra} {det['similitud']:.4f}  {det['titulo']}")


La métrica de relevancia por similitud coseno es simple pero efectiva como proxy de la calidad del retrieval. Tiene una limitación conocida: puede ser alta incluso si el chunk habla del tema correcto pero no responde exactamente la pregunta. Para producción, se complementa con métricas más sofisticadas como Context Precision (¿qué fracción del contexto es realmente útil?) y Context Recall (¿qué fracción de la información necesaria está en el contexto recuperado?) del framework RAGAS.

**Celda 23: Detección de alucinaciones con LLM-as-judge**

In [ ]:
def detectar_alucinacion(pregunta: str, respuesta: str, contexto: str) -> dict:
    """
    LLM-as-judge: evalúa si la respuesta está soportada por el contexto.

    Pasos:
    1. Construye un prompt_juez que incluya CONTEXTO, PREGUNTA y RESPUESTA
       y pida al modelo devolver JSON con:
       {"soportada": bool, "confianza": float 0-1,
        "elementos_no_soportados": [list of strings]}
    2. Llama a client.responses.create con system prompt de evaluador
    3. Limpia posibles bloques ```json con re.sub
    4. Parsea con json.loads y devuelve el dict
       (en caso de error, devuelve dict con raw y error)
    """
    ### EMPIEZA TU CÓDIGO ###
    # TU CÓDIGO AQUÍ
    raise NotImplementedError("Completa esta celda")
    ### FIN DE TU CÓDIGO ###


# Aplicar al resultado del pipeline
contexto_eval = construir_contexto(res_eval["chunks_finales"])
juicio = detectar_alucinacion(pregunta_eval, res_eval["respuesta"], contexto_eval)

print(f"=== DETECCIÓN DE ALUCINACIÓN (LLM-as-judge) ===")
print(f"Soportada  : {juicio.get('soportada')}")
print(f"Confianza  : {juicio.get('confianza')}")
elementos = juicio.get("elementos_no_soportados", [])
if elementos:
    print("Elementos no soportados:")
    for elem in elementos:
        print(f"  - {elem}")
else:
    print("Elementos no soportados: ninguno ✓")


LLM-as-judge para detección de alucinación (también llamada "faithfulness" en el framework RAGAS) evalúa si cada afirmación de la respuesta tiene respaldo explícito en el contexto. El prompt fuerza una respuesta JSON estructurada para que sea parseada programáticamente. En sistemas en producción, se usa `gpt-4o` (no mini) como juez para mayor precisión, y se promedian múltiples evaluaciones para reducir la varianza del juez.

**Celda 24: Medición de coste y latencia**

In [ ]:
def medir_coste_y_latencia(pregunta: str) -> dict:
    """
    Ejecuta el pipeline RAG completo midiendo latencia total y coste económico.

    Precios gpt-4o-mini (Mayo 2024):
        Input tokens : $0.15 / 1M tokens
        Output tokens: $0.60 / 1M tokens

    Args:
        pregunta: Pregunta a procesar.

    Returns:
        Dict con: pregunta, latencia_s, input_tokens, output_tokens, cost_usd, respuesta.
    """
    PRECIO_INPUT_POR_MILLON  = 0.15   # USD
    PRECIO_OUTPUT_POR_MILLON = 0.60   # USD

    t0 = time.time()

    # Retrieval híbrido
    res_sem  = busqueda_semantica(pregunta, n=7)
    res_bm25 = busqueda_bm25(pregunta, n=7)
    res_rrf  = fusion_rrf(res_sem, res_bm25, k=60)
    chunks_finales = reranking(pregunta, res_rrf[:7], top_k=3)

    # Construir mensaje para la generación
    contexto = construir_contexto(chunks_finales)
    system_prompt = (
        "Eres un asistente experto en IA. Responde ÚNICAMENTE usando el CONTEXTO. "
        "Para cada afirmación cita la fuente con [N]. "
        "Si no hay información suficiente, di 'No tengo información sobre esto en los documentos disponibles.' "
        "NO inventes datos."
    )
    user_message = f"{contexto}\n\nPREGUNTA: {pregunta}"

    # Generación (capturamos el objeto response completo para extraer usage)
    response = client.responses.create(
        model="gpt-4o-mini",
        input=[
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_message}
        ]
    )

    latencia_total = time.time() - t0

    input_tokens  = response.usage.input_tokens
    output_tokens = response.usage.output_tokens
    cost_usd = (
        (input_tokens  / 1_000_000) * PRECIO_INPUT_POR_MILLON +
        (output_tokens / 1_000_000) * PRECIO_OUTPUT_POR_MILLON
    )

    return {
        "pregunta": pregunta[:60] + "..." if len(pregunta) > 60 else pregunta,
        "latencia_s": round(latencia_total, 2),
        "input_tokens": input_tokens,
        "output_tokens": output_tokens,
        "total_tokens": input_tokens + output_tokens,
        "cost_usd": round(cost_usd, 6),
        "respuesta": response.output_text
    }


# Prueba de medición de coste y latencia
print("=== MEDICIÓN DE COSTE Y LATENCIA ===")
metricas_costes = medir_coste_y_latencia(preguntas_demo[1])

print(f"Pregunta      : {metricas_costes['pregunta']}")
print(f"Latencia total: {metricas_costes['latencia_s']}s")
print(f"Input tokens  : {metricas_costes['input_tokens']:,}")
print(f"Output tokens : {metricas_costes['output_tokens']:,}")
print(f"Total tokens  : {metricas_costes['total_tokens']:,}")
print(f"Coste estimado: ${metricas_costes['cost_usd']:.6f} USD")
print(f"Coste por 1000 preguntas: ${metricas_costes['cost_usd'] * 1000:.4f} USD")
print(f"\nRespuesta:")
print(metricas_costes["respuesta"])


Medir el coste por consulta es fundamental para la viabilidad económica de un sistema RAG en producción. Con gpt-4o-mini a $0.15/1M tokens de entrada y $0.60/1M de salida, una consulta RAG típica (500 tokens de contexto + 200 de pregunta + 300 de respuesta) cuesta menos de $0.001 USD. A escala de 100,000 consultas/día, el coste diario sería del orden de $100 USD, perfectamente viable para aplicaciones empresariales. La latencia total incluye el tiempo de embedding (dos llamadas: pregunta + chunks), el reranking (CPU local) y la generación.

**Celda 25: Benchmark completo**

In [ ]:
# Definir 5 pares pregunta-hechos clave esperados
benchmark_preguntas = [
    {
        "pregunta": "¿Qué es el aprendizaje supervisado?",
        "hechos_clave": ["datos etiquetados", "función de pérdida", "predicciones"]
    },
    {
        "pregunta": "¿Cómo funciona la retropropagación?",
        "hechos_clave": ["gradiente", "backpropagation", "pesos", "regla de la cadena"]
    },
    {
        "pregunta": "¿Qué diferencia hay entre fine-tuning y RAG?",
        "hechos_clave": ["fine-tuning", "reentrenamiento", "recuperación", "alucinaciones"]
    },
    {
        "pregunta": "¿Cuáles son los riesgos del sesgo algorítmico?",
        "hechos_clave": ["sesgo", "discriminación", "COMPAS", "datos de entrenamiento"]
    },
    {
        "pregunta": "¿Qué es BM25 y para qué sirve?",
        "hechos_clave": ["recuperación", "TF-IDF", "frecuencia", "léxica"]
    }
]

print("Ejecutando benchmark completo (5 preguntas)...")
print("Esto puede tardar 1-2 minutos...\n")

filas_benchmark = []

for item in benchmark_preguntas:
    pregunta = item["pregunta"]
    hechos = item["hechos_clave"]

    print(f"  Procesando: {pregunta[:55]}...")

    # Medir coste y latencia (incluye pipeline completo)
    metricas = medir_coste_y_latencia(pregunta)

    # Ejecutar pipeline separado para obtener chunks finales para evaluar relevancia
    res_pip = pipeline_rag_completo(pregunta, n_candidatos=7, top_k=3)

    # Evaluar relevancia de recuperación
    rel = evaluar_relevancia(pregunta, res_pip["chunks_finales"])

    # Calcular cobertura de hechos clave (qué fracción aparece en la respuesta)
    respuesta_lower = metricas["respuesta"].lower()
    hechos_encontrados = sum(1 for h in hechos if h.lower() in respuesta_lower)
    cobertura_hechos = round(hechos_encontrados / len(hechos), 2)

    # Juicio de fidelidad (LLM-as-judge simplificado)
    contexto_pip = construir_contexto(res_pip["chunks_finales"])
    juicio = detectar_alucinacion(pregunta, metricas["respuesta"], contexto_pip)

    filas_benchmark.append({
        "Pregunta": pregunta[:45] + "...",
        "Latencia (s)": metricas["latencia_s"],
        "Input tok": metricas["input_tokens"],
        "Output tok": metricas["output_tokens"],
        "Coste ($)": metricas["cost_usd"],
        "Relevancia": rel["relevancia_media"],
        "Cob. hechos": cobertura_hechos,
        "Fiel.": juicio.get("soportada", "?"),
        "Conf. juez": juicio.get("confianza", 0.0)
    })

df_benchmark = pd.DataFrame(filas_benchmark)

print("\n" + "="*70)
print("RESULTADOS DEL BENCHMARK")
print("="*70)
print(df_benchmark.to_string(index=False))

print("\n── Promedios ──")
cols_num = ["Latencia (s)", "Input tok", "Output tok", "Coste ($)", "Relevancia", "Cob. hechos", "Conf. juez"]
promedios = df_benchmark[cols_num].mean()
for col, val in promedios.items():
    print(f"  {col:20s}: {val:.4f}")

print(f"\n  Coste total benchmark: ${df_benchmark['Coste ($)'].sum():.6f} USD")


El benchmark sistematiza la evaluación en cuatro dimensiones: (1) **Latencia**: tiempo total del pipeline incluyendo embedding, retrieval, reranking y generación; (2) **Coste**: en USD por consulta, extrapolable a escala; (3) **Relevancia de recuperación**: similitud coseno pregunta-chunks; (4) **Fidelidad**: si la respuesta está soportada por el contexto (LLM-as-judge). La cobertura de hechos clave es una métrica adicional que verifica que la respuesta menciona los conceptos esperados, útil cuando hay ground truth disponible.